In [4]:
import sys
sys.path.append(r'C:\Users\Nikhil Kapoor\Documents\dataScienceProjects\src')
from tennis_stats import win_rate_on, head_to_head

from importlib import reload
import tennis_stats; reload(tennis_stats)
from tennis_stats import win_rate_on, head_to_head

import json, os
os.makedirs('data', exist_ok=True)


In [5]:
import pandas as pd
URL = 'https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_2010.csv'
df = pd.read_csv(URL)

In [6]:
import sys
sys.path.append(r'C:\Users\Nikhil Kapoor\Documents\dataScienceProjects\src')
from tennis_stats import win_rate_on, head_to_head

In [7]:
unique_players = set(df['winner_name']).union(set(df['loser_name']))
print(df.shape, len(unique_players), 'unique_players')



(3030, 49) 472 unique_players


In [8]:
top10 = (df.groupby('winner_name')
         .size()
         .sort_values(ascending=False)
         .head(10)
         .reset_index())
top10.columns = ['winner_name','wins']

In [9]:
wins = (df.groupby('winner_name')
       .size()
       .rename('wins'))
losses = (df.groupby('loser_name')
       .size()
       .rename('losses'))
aces = (df.groupby('winner_name')['w_ace']
       .sum()
       .rename('aces'))
avg_min = (df.groupby('winner_name')['minutes']
       .mean()
       .rename('avg_min_won'))


In [10]:
stats = pd.concat([wins,losses,aces,avg_min], axis = 1).fillna(0)
stats['win_rate'] = stats['wins']/ (stats['wins'] + stats['losses'])
stats = stats.sort_values('wins', ascending=False).head(10)

records = stats.reset_index().to_dict(orient='records')

with open('data/top10_stats_2010.json', 'w') as f:
    json.dump(records, f, indent=2, default=str)

In [11]:
with open('data/top10_stats_2010.json') as f:
            loaded = json.load(f)

top_ace = max(loaded, key=lambda r: r['aces'])
print(f"{top_ace['index']}: {int(top_ace['aces'])} aces")


Andy Roddick: 610 aces


In [12]:
top10_names = stats.index.tolist()
pivot = pd.DataFrame({
    'Hard' : [win_rate_on(df, p, 'Hard') for p in top10_names],
    'Clay' : [win_rate_on(df, p, 'Clay') for p in top10_names],
    'Grass' : [win_rate_on(df, p, 'Grass') for p in top10_names],
}, index=top10_names
)

for name in top10_names:
    overall = win_rate_on(df, name)
    clay = win_rate_on(df, name, 'Clay')
    print(f"{name:25s} overall= {overall:.2f} clay={clay:.2f}")


Rafael Nadal              overall= 0.88 clay=1.00
Roger Federer             overall= 0.84 clay=0.71
Novak Djokovic            overall= 0.78 clay=0.75
David Ferrer              overall= 0.71 clay=0.82
Robin Soderling           overall= 0.72 clay=0.70
Jurgen Melzer             overall= 0.68 clay=0.68
Andy Roddick              overall= 0.73 clay=0.67
Gael Monfils              overall= 0.70 clay=0.67
Andy Murray               overall= 0.72 clay=0.60
Tomas Berdych             overall= 0.63 clay=0.78


In [13]:
import os
os.makedirs('data', exist_ok=True)
stats.to_csv('data/top10_stats_2010.csv')
# Add data/ to .gitignore if it isn't already — raw CSVs shouldn't live in git.

In [14]:
spread = pivot.max(axis=1) - pivot.min(axis=1)
spread.sort_values(ascending=False).head(3)


Tomas Berdych    0.313665
Gael Monfils     0.220000
David Ferrer     0.196742
dtype: float64

29-05-2026

In [16]:
fed_all = df[(df['winner_name'] == 'Roger Federer') | (df['loser_name'] == 'Roger Federer')]
fed_longest = fed_all.sort_values('minutes', ascending=False).head(1)
fed_longest[['tourney_name','round','winner_name','loser_name','surface','minutes']]

,tourney_name,round,winner_name,loser_name,surface,minutes
2462,US Open,SF,Novak Djokovic,Roger Federer,Hard,224.0


In [17]:

print(head_to_head(df, 'Rafael Nadal', 'Roger Federer'))
print(head_to_head(df, 'Rafael Nadal', 'Novak Djokovic'))


(1, 1)
(2, 0)


In [18]:
try:
    win_rate_on(df, 'Nadal','Clay')
except ValueError as e:
    print(f"Caught: {e}")

try:
    head_to_head(df, 'Rafael Nadal', 'Jannik Sinner')
except ValueError as e:
    print(f"Caught: {e}")

Caught: Player Nadal not found in 2010 dataset
Caught: Player Jannik Sinner not found in 2010 dataset.


In [19]:
big_four = ['Rafael Nadal', 'Roger Federer', 'Andy Murray', 'Noval Djokovic']

pd.pivot_table(
    df[df['winner_name'].isin(big_four)],
    values='minutes',
    index='winner_name',
    columns='surface',
    aggfunc='mean'
)


surface,Clay,Grass,Hard
winner_name,,,
Andy Murray,138.500000,115.666667,101.529412
Rafael Nadal,107.954545,137.111111,118.725000
Roger Federer,100.700000,102.375000,92.744681


In [20]:
reload(tennis_stats)
from tennis_stats import top_players_by_wins

for name, wins in top_players_by_wins(df):
    print(f"{name:25s} {wins}")

Rafael Nadal              71
Roger Federer             66
Novak Djokovic            63
David Ferrer              60
Robin Soderling           57
Jurgen Melzer             53
Andy Roddick              48
Gael Monfils              46
Andy Murray               46
Tomas Berdych             45


**Start of Numpy Practice**

In [38]:
import numpy as np

In [42]:
minutes = df['minutes'].dropna().to_numpy()
print(type(minutes))
print(minutes.shape)
print(minutes.dtype)
print(minutes[:5])

<class 'numpy.ndarray'>
(2686,)
float64
[ 84.  70. 121.  64.  69.]


In [44]:
hours_loop = [m/60 for m in minutes]
hours_vec = minutes/60

print(hours_loop[:5])
print(hours_vec[:5])
print(np.allclose(hours_loop, hours_vec))

%timeit [m/60 for m in minutes]
%timeit minutes/60

[np.float64(1.4), np.float64(1.1666666666666667), np.float64(2.0166666666666666), np.float64(1.0666666666666667), np.float64(1.15)]
[1.4        1.16666667 2.01666667 1.06666667 1.15      ]
True
153 μs ± 803 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
1.61 μs ± 9.99 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [50]:
avg = minutes.mean()
diff = minutes - avg
print(diff[:5])
print(f"Range: {diff.min():.0f} to {diff.max():.0f} min from average")

longest_idx = np.argmax(diff)
print(f"Match index {longest_idx} ran {diff[longest_idx]:.0f} min over average")

[-23.03425168 -37.03425168  13.96574832 -43.03425168 -38.03425168]
Range: -102 to 558 min from average
Match index 1562 ran 558 min over average


In [58]:
long = minutes > 180
print(long[:5])
print(long.sum())
print(minutes[long].mean())
print(minutes[long])

[False False False False False]
148
216.57432432432432
[191. 185. 202. 193. 190. 293. 216. 201. 235. 188. 182. 184. 197. 237.
 236. 231. 201. 249. 206. 200. 228. 257. 231. 185. 212. 202. 200. 181.
 210. 237. 213. 205. 278. 232. 230. 182. 191. 194. 202. 181. 205. 184.
 198. 205. 198. 214. 244. 218. 196. 296. 226. 215. 223. 199. 185. 216.
 214. 225. 196. 245. 256. 192. 210. 191. 184. 211. 278. 207. 188. 253.
 186. 196. 242. 255. 207. 198. 185. 227. 188. 253. 231. 214. 216. 201.
 193. 246. 665. 197. 257. 252. 216. 235. 205. 218. 212. 233. 205. 227.
 184. 195. 225. 184. 276. 183. 188. 192. 196. 188. 212. 223. 187. 239.
 228. 200. 182. 186. 187. 199. 207. 220. 232. 218. 194. 193. 219. 197.
 228. 253. 186. 231. 198. 299. 215. 236. 198. 222. 213. 263. 268. 185.
 240. 224. 223. 182. 191. 186. 186. 192.]


In [74]:
def win_rate_np(df, player, surface=None):
    """Faster win_rate using NumPy boolean ops on the underlying arrays."""
    winners = df['winner_name'].to_numpy()
    losers  = df['loser_name'].to_numpy()
    wins_mask   = (winners == player)
    losses_mask = (losers  == player)
    if surface is not None:
        surfs = df['surface'].to_numpy()
        wins_mask   &= (surfs == surface)
        losses_mask &= (surfs == surface)
    wins   = wins_mask.sum()
    losses = losses_mask.sum()
    return wins / (wins + losses) if (wins + losses) > 0 else 0.0

print(win_rate_np(df, 'Rafael Nadal', 'Clay'))   # ~0.95
print(win_rate_np(df, 'Rafael Nadal'))            # overall

1.0
0.8765432098765432


In [76]:
reload(tennis_stats)
from tennis_stats import match_length_zscore

In [82]:
z = match_length_zscore(df)
print(f"Max z: {z.max():.2f}, min:{z.min():.2f}")
print(f"Matches >3 sd from mean: {(np.abs(z) > 3).sum()}")

Max z: 13.24, min:-2.42
Matches >3 sd from mean: 30
